In [2]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, '..')   # 让 Python 能找到上一层文件夹里的 common 包

import pandas as pd
from common.data import load_raw
from common.interface import model_folds

raw = load_raw('../amp-parkinsons-disease-progression-prediction')
samples = pd.read_csv('../artifacts/shared/samples.csv')
patients = pd.read_csv('../artifacts/shared/patient_folds.csv')

folds = list(model_folds(raw, samples, patients))   # 把五折全部构建出来，第一次跑约 1 分钟
print("五折都准备好了：", len(folds))

五折都准备好了： 5


In [3]:
data = folds[0]   # 第 0 折：用其余四折患者训练、这折患者验证

print("训练特征表形状:", data.train_features.shape)
print("验证特征表形状:", data.valid_features.shape)
data.train_features.head()

训练特征表形状: (9928, 1202)
验证特征表形状: (2769, 1202)


,visit_id,patient_id,visit_month,horizon,target,target_month,prediction_id,protein::O00391,protein::O00533,protein::O00584,...,peptide::Q9UBR2::NVDGVNYASITR,peptide::P10909::ELDESLQVAER,peptide::P13611::AQC(UniMod_4)GGGLLGVR,peptide::Q12841::GAQTQTEEEMTR,peptide::O00584::ELDLNSVLLK,peptide::P02647::PALEDLR,peptide::P05408::VSEADIQR,peptide::P07711::NSWGEEWGMGGYVK,peptide::Q13740::SSNTYTLTDVRR,peptide::P01857::EPQVYTLPPSRDELTK
0,10053_0,10053,0,0,updrs_1,0,10053_0_updrs_1_plus_0_months,9104.27,402321.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10053_0,10053,0,12,updrs_1,12,10053_0_updrs_1_plus_12_months,9104.27,402321.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10053_0,10053,0,24,updrs_1,24,10053_0_updrs_1_plus_24_months,9104.27,402321.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10053_0,10053,0,6,updrs_1,6,10053_0_updrs_1_plus_6_months,9104.27,402321.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10053_0,10053,0,0,updrs_2,0,10053_0_updrs_2_plus_0_months,9104.27,402321.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 看看缺失有多严重
miss = data.train_features.isna().mean().sort_values(ascending=False)
print("缺失最多的 5 列:\n", miss.head())
print("\n整体缺失比例:", data.train_features.isna().mean().mean().round(3))

"错误案例"

In [ ]:
from sklearn.linear_model import Ridge
from common.metric import smape_plus_one

# 1. 把"地址信息"列扔掉，只留下模型该吃的特征
KEY_COLS = ["visit_id", "patient_id", "visit_month", "horizon", "target", "target_month", "prediction_id"]
X_train = data.train_features.drop(columns=KEY_COLS)
X_valid = data.valid_features.drop(columns=KEY_COLS)

# 2. 缺失值填充：中位数从【训练折】算，验证集照抄
med = X_train.median()
X_train = X_train.fillna(med)
X_valid = X_valid.fillna(med)

# 3. target 是 updrs_1..4 文字，编码成 0..3 让模型知道"这行要预测哪一项评分"
cats = ["updrs_1", "updrs_2", "updrs_3", "updrs_4"]
X_train["target_code"] = pd.Categorical(data.train_features["target"], categories=cats).codes
X_valid["target_code"] = pd.Categorical(data.valid_features["target"], categories=cats).codes

# 4. 标签：真实评分
y_train = data.train_samples["rating"]
y_valid = data.valid_samples["rating"]

# 5. 训练岭回归（线性回归 + 防系数乱晃的惩罚）
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

# 6. 预测并裁掉负数（UPDRS 评分不可能是负的，这是合理的后处理）
pred = model.predict(X_valid)
pred = pred.clip(min=0)

# 7. 算这一折的 SMAPE+1，越低越好
score = smape_plus_one(y_valid, pred)
print("第 0 折 SMAPE+1:", round(score, 2))

"修正"

In [4]:
KEY_COLS = ["visit_id", "patient_id", "visit_month", "horizon", "target", "target_month", "prediction_id"]

def prep_features(d):
    Xtr = d.train_features.drop(columns=KEY_COLS).copy()
    Xva = d.valid_features.drop(columns=KEY_COLS).copy()
    for df in (Xtr, Xva):
        for c in df.columns:
            if c.startswith(("protein::", "peptide::")):
                df[c] = np.log1p(df[c])                    # 修复①：强度值跨数量级 → 取对数
    Xtr = pd.concat([Xtr, pd.get_dummies(d.train_features["target"], prefix="t").astype(float)], axis=1)
    Xva = pd.concat([Xva, pd.get_dummies(d.valid_features["target"], prefix="t").astype(float)], axis=1)  # 修复②：独热编码
    med = Xtr.median()
    return Xtr.fillna(med), Xva.fillna(med)                # 修复③：缺失值=训练折中位数

baseline

In [5]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from common.metric import evaluate_oof
import time


t0 = time.time()
oof = []
for d in folds:
    Xtr, Xva = prep_features(d)
    meta = [c for c in Xtr.columns if c.startswith("t_updrs") or c in ("visit_month", "horizon")]
    sc = StandardScaler().fit(Xtr[meta])                   # 标准化（修复④，虽然这3列量级不大，养成习惯）
    m = Ridge(alpha=1.0).fit(sc.transform(Xtr[meta]), d.train_samples["rating"])
    pred = np.clip(m.predict(sc.transform(Xva[meta])), 0, None)   # 裁掉负数
    oof.append(pd.DataFrame({"prediction_id": d.valid_features["prediction_id"], "rating": pred}))
ridge_sec = round(time.time() - t0, 1)

eval_ridge = evaluate_oof(samples, pd.concat(oof))
print("Baseline Ridge 五折结果:", eval_ridge)

Baseline Ridge 五折结果: {'fold_scores': {0: 66.12651056694682, 1: 72.69165867936398, 2: 71.23513610394212, 3: 76.21275508755991, 4: 71.61830463299174}, 'mean_score': 71.57687301416091, 'std_score': 3.2421937381547967, 'oof_score': 71.41215420712409, 'n_predictions': 12697}


lightGBM"改进"

In [6]:
import lightgbm as lgb, re

t0 = time.time()
oof = []
for d in folds:
    Xtr, Xva = prep_features(d)
    ren = lambda c: re.sub(r'[^0-9A-Za-z_]+', '_', c)   # LightGBM 不认列名里的 ::，先改名
    Xtr = Xtr.rename(columns=ren); Xva = Xva.rename(columns=ren)
    m = lgb.train(dict(objective="regression", verbosity=-1, seed=42,
                       learning_rate=0.05, num_leaves=63, min_data_in_leaf=50),
                  lgb.Dataset(Xtr, label=d.train_samples["rating"]), num_boost_round=400)
    pred = np.clip(m.predict(Xva), 0, None)
    oof.append(pd.DataFrame({"prediction_id": d.valid_features["prediction_id"], "rating": pred}))
lgbm_sec = round(time.time() - t0, 1)

eval_lgbm = evaluate_oof(samples, pd.concat(oof))
print("LightGBM 五折结果:", eval_lgbm)

LightGBM 五折结果: {'fold_scores': {0: 63.65343522928484, 1: 73.26050832364997, 2: 71.6296626467955, 3: 74.37765273578039, 4: 73.23666881061405}, 'mean_score': 71.23158554922495, 'std_score': 3.8889960889469335, 'oof_score': 71.03580814697659, 'n_predictions': 12697}


改进

In [7]:
t0 = time.time()
oof = []
for d in folds:
    Xtr, Xva = prep_features(d)
    ren = lambda c: re.sub(r'[^0-9A-Za-z_]+', '_', c)
    Xtr = Xtr.rename(columns=ren); Xva = Xva.rename(columns=ren)
    m = lgb.train(dict(objective="regression", verbosity=-1, seed=42,
                       learning_rate=0.03,          # 改①：小步慢学
                       num_leaves=63, min_data_in_leaf=50,
                       feature_fraction=0.5),       # 改②：每棵树只看随机一半特征，防噪声列带偏
                  lgb.Dataset(Xtr, label=d.train_samples["rating"]),
                  num_boost_round=800)              # 改③：树多近四倍，配低学习率
    pred = np.clip(m.predict(Xva), 0, None)
    oof.append(pd.DataFrame({"prediction_id": d.valid_features["prediction_id"], "rating": pred}))
lgbm2_sec = round(time.time() - t0, 1)

eval_lgbm2 = evaluate_oof(samples, pd.concat(oof))
print("LightGBM 调参版 五折结果:", eval_lgbm2)

LightGBM 调参版 五折结果: {'fold_scores': {0: 62.08203301221469, 1: 72.1158932116846, 2: 69.93274876251519, 3: 73.42039332029768, 4: 72.32965941469763}, 'mean_score': 69.97614554428196, 'std_score': 4.106075652264397, 'oof_score': 69.76541540784194, 'n_predictions': 12697}
